# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omi290/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

# Finding 1 – The Anatomy of Growing Content

The paper reports that growing content tends to be longer, younger, and slightly better positioned in search than declining content. The comparison is based on large portfolio-level groups and is presented as an observational comparison rather than a causal result. :contentReference[oaicite:0]{index=0}

### Methodology Question

How was the "growing" versus "declining" label assigned? Was the trend measured using only historical information available before evaluation, and would the same relationship remain when validating on different clients or future time periods?

This question checks whether the observed relationship generalizes beyond the current portfolio.

# Finding 2 – AI Traffic: A Different Signal

The paper reports that pages receiving AI referrals behave differently from traditional organic pages while also emphasizing that AI traffic represents only about 1% of tracked sessions and should not be overinterpreted. :contentReference[oaicite:1]{index=1}

### Methodology Question

Were differences between AI-referred pages and traditional pages evaluated after controlling for other factors such as page age, content length, or search visibility?

This would help determine whether AI referrals themselves explain the observed differences or whether other characteristics contribute to the pattern.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

# Honest Validation

My Week 5 work focused on exploratory correlation and feature analysis rather than supervised prediction.

To validate the findings, I repeated the analysis using the same historical March 2026 warehouse data while ensuring only observations with valid GA4 measurements were included.

The feature set remained unchanged and no future information or label-derived variables were introduced.

In [1]:
!pip -q install duckdb huggingface_hub pyarrow scikit-learn

In [2]:
from google.colab import userdata
from huggingface_hub import hf_hub_download
import duckdb
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

In [3]:
HF_TOKEN = userdata.get("HF_TOKEN")

In [4]:
march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=HF_TOKEN
)

print(march_path)

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet


In [5]:
db = duckdb.connect()

In [6]:
db.sql(f"""
CREATE OR REPLACE TABLE march_data AS
SELECT *
FROM read_parquet('{march_path}')
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [7]:
feature_vector = db.sql("""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_pageviews,
    ga4_sessions,
    scroll_events
FROM march_data
WHERE ga4_data_available IS TRUE
""").df()

feature_vector.head()

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions,scroll_events
0,2026-03-01,client_65de48885f4ef01b,content_09be8cc7fcb222af,0,0,NaN,1,1,0
1,2026-03-01,client_65de48885f4ef01b,content_851afac9fe13612e,0,0,NaN,1,1,0
2,2026-03-01,client_65de48885f4ef01b,content_cee6c6fc8c51af14,0,0,NaN,1,1,0
3,2026-03-01,client_65de48885f4ef01b,content_5e120e972f11f833,0,0,NaN,1,1,0
4,2026-03-01,client_65de48885f4ef01b,content_16a7291bb6ecaebe,0,0,NaN,1,1,0


In [8]:
print("Rows before validation:", len(feature_vector))

validated = feature_vector[
    feature_vector["ga4_sessions"] > 0
]

print("Rows after validation:", len(validated))

feature_vector.describe()

validated.describe()

Rows before validation: 413966
Rows after validation: 410335


,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions,scroll_events
count,410335,410335.000000,410335.000000,361095.000000,410335.000000,410335.000000,410335.000000
mean,2026-03-18 06:09:14.035117,206.655099,0.957754,14.333130,3.608235,3.167675,0.534444
min,2026-03-01 00:00:00,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000
25%,2026-03-12 00:00:00,16.000000,0.000000,4.613636,1.000000,1.000000,0.000000
50%,2026-03-18 00:00:00,76.000000,0.000000,9.464968,2.000000,1.000000,0.000000
75%,2026-03-25 00:00:00,218.000000,1.000000,21.510163,3.000000,3.000000,1.000000
max,2026-03-31 00:00:00,39305.000000,274.000000,377.000000,875.000000,792.000000,254.000000
std,NaN,495.138387,2.922455,13.059321,8.006351,7.417227,1.602421


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

# Leakage Audit

I reviewed every feature used in the Week 5 analysis.

Checks performed:

- client_hash_id excluded from analysis
- content_hash_id excluded from analysis
- report_date not used as a predictive feature
- no future-window information included
- no label-derived variables included

The remaining features represent historical search and engagement measurements that would have been available before analysis.

No obvious leakage was observed.

In [9]:
feature_vector.columns

Index(['report_date', 'client_hash_id', 'content_hash_id', 'gsc_impressions',
       'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions',
       'scroll_events'],
      dtype='object')

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

# Original Claim

The analysis predicts which content should be refreshed.

# Revised Claim

The analysis observed relationships among historical search and engagement metrics that may support content refresh prioritization. These observations are directional and intended for decision-support rather than causal prediction.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.